### Imports

In [71]:
from ultralytics import YOLO
import numpy as np
from pathlib import Path
from tqdm import tqdm
import cv2
import os
import matplotlib.pyplot as plt
import random
from ultralytics.trackers.track import register_tracker

### Load Model

In [39]:
MODEL = "yolov8n"
TRACKER = "botsort.yaml"

In [46]:
model = YOLO(f'{MODEL}.pt')
model.eval()

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_s

### Load Dataset

In [91]:
MOT20DIRTEST = "../data/MOT20/test"
TRAINSET = "../data/yolo_mot_dataset"
OUTPUT = f"../output/{MODEL}"
SEQ_PATH =  "../data/MOT20/test/MOT20-08/img1"
DATACONFIG = "../data/"

### Visualize

In [86]:
img_files = sorted(os.listdir(SEQ_PATH))
first_frame_path = os.path.join(SEQ_PATH, img_files[0])
frame = cv2.imread(first_frame_path)
height, width, channels = frame.shape

In [90]:
# Loop through the video frames
def create_video(model, img_files, seq_path, video_dir, output_name):
    if not os.path.exists(video_dir):
        os.makedirs(video_dir)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    fps = 24  # Adjust frames per second as needed.
    video_writer = cv2.VideoWriter(os.path.join(video_dir, f"MOT20-08{output_name}.mp4"), fourcc, fps, (width, height))
    for idx, img_name in enumerate(tqdm(img_files, desc=f"MOT20-08")):
        img_path = os.path.join(seq_path, img_name)
        frame = cv2.imread(img_path)
        if frame is None:
            print(f"Warning: Could not read {img_path}")
            continue
        result = model.track(frame, persist=True, verbose = False)[0]
        annotated_frame = result.plot()
        video_writer.write(annotated_frame)
    video_writer.release()


In [ ]:
video_dir = os.path.join(OUTPUT,"videos")    
create_video(model, img_files, SEQ_PATH, video_dir)

In [10]:
def visualize_predictions(model, dataset_path, num_images=5):
    """
    Visualize predictions from a YOLOv8 model on a subset of images from the dataset.

    Parameters:
    - model: A loaded YOLOv8 model (e.g., YOLO("yolov8s.pt")).
    - dataset_path: Root folder of your dataset. The function will search recursively for .jpg images.
    - num_images: Number of random images to visualize.
    """
    # Recursively collect all JPG images from the dataset_path.
    all_images = []
    for root, _, files in os.walk(dataset_path):
        for file in files:
            if file.endswith(".jpg"):
                all_images.append(os.path.join(root, file))

    if len(all_images) == 0:
        print("No JPG images found in the provided dataset path.")
        return

    # Randomly select a few images for visualization.
    sample_images = random.sample(all_images, min(num_images, len(all_images)))

    # Loop over the selected images.
    for img_path in sample_images:
        # Read the image using OpenCV.
        img = cv2.imread(img_path)
        if img is None:
            continue

        # Run YOLOv8 inference.
        results = model(img, verbose=False)[0]
        boxes = results.boxes  # List of detected boxes

        # Draw bounding boxes and confidence scores on a copy of the image.
        img_vis = img.copy()
        for box in boxes:
            # Extract the bounding box coordinates.
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            conf = float(box.conf[0])
            # Draw a rectangle around the detection.
            cv2.rectangle(img_vis, (int(x1), int(y1)), (int(x2), int(y2)), color=(0, 255, 0), thickness=2)
            # Write the confidence score above the detection.
            cv2.putText(img_vis, f"{conf:.2f}", (int(x1), int(y1)-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), thickness=2)

        # Convert the image from BGR to RGB for displaying with matplotlib.
        img_rgb = cv2.cvtColor(img_vis, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(10, 8))
        plt.imshow(img_rgb)
        plt.title(f"Predictions on: {os.path.basename(img_path)}")
        plt.axis("off")
        plt.show()


In [ ]:
model.reset_callbacks()
visualize_predictions(model, MOT20DIRTEST, num_images=5)

### Evaluation Code

In [77]:
def evaluate(model, dataset_path, output_dir):
    model.eval()
    os.makedirs(output_dir, exist_ok=True)
    for seq in os.listdir(dataset_path):
        seq_path = os.path.join(dataset_path, seq, "img1")
        if not os.path.isdir(seq_path):
            continue
        print("Resetting Tracker")
        model.reset_callbacks()
        register_tracker(model, persist=True)
        print(f"Processing sequence: {seq}")
        output_file = os.path.join(output_dir, f"{seq}.txt")
        with open(output_file, "w") as f_out:
            # Sort frames
            img_files = sorted(os.listdir(seq_path))
            for _, img_name in enumerate(tqdm(img_files, desc=f"{seq}")):
                if not img_name.endswith(".jpg"):
                    continue
                frame_num = int(os.path.splitext(img_name)[0])
                img_path = os.path.join(seq_path, img_name)
                img = cv2.imread(img_path)

                results = model.track(
                    source=img, 
                    tracker = "bytetrack.yaml",
                    persist=True, 
                    verbose=False,   
                )[0]              
                boxes = results.boxes

                for box in boxes:
                    x1, y1, x2, y2 = box.xyxy[0].tolist()
                    conf = float(box.conf[0])
                    width = x2 - x1
                    height = y2 - y1
                    if box.id is not None:
                        track_id = int(box.id[0])  # Access the first element if it's a tensor

                    else:
                        continue  
    
                    mot_line = f"{frame_num},{track_id},{x1:.2f},{y1:.2f},{width:.2f},{height:.2f},{conf:.4f},-1,-1,-1\n"
                    f_out.write(mot_line)


    return results

### Evaluate Baseline

In [78]:
output_dir = os.path.join(OUTPUT, "baseline")
evaluate(model, MOT20DIRTEST, output_dir)

Resetting Tracker
Processing sequence: MOT20-04


MOT20-04: 100%|██████████| 2080/2080 [01:36<00:00, 21.59it/s]


Resetting Tracker
Processing sequence: MOT20-07


MOT20-07: 100%|██████████| 585/585 [00:26<00:00, 22.48it/s]


Resetting Tracker
Processing sequence: MOT20-08


MOT20-08: 100%|██████████| 806/806 [00:27<00:00, 29.24it/s]


Resetting Tracker
Processing sequence: MOT20-06


MOT20-06: 100%|██████████| 1008/1008 [00:35<00:00, 28.22it/s]


ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
keypoints: None
masks: None
names: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted plant',

### Transform Images

In [ ]:
results = os.path.join(OUTPUT, "yolov8nbest/results")
results = model.train(data="../data/yolo_mot_dataset/data.yaml", project = )

New https://pypi.org/project/ultralytics/8.3.107 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.105 🚀 Python-3.11.8 torch-2.6.0 CPU (Apple M2 Max)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=../data/yolo_mot_dataset/data.yaml, epochs=100, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train2, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, 

Fontconfig warning: ignoring UTF-8: not a valid region tag


Overriding model.yaml nc=80 with nc=12

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytic

train: Scanning /Users/ibraheemalsaghier/Desktop/Personal Projects/yolo/data/yolo_mot_dataset/labels/train.cache... 2851 images, 0 backgrounds, 0 corrupt: 100%|██████████| 2851/2851 [00:00<?, ?it/s]
val: Scanning /Users/ibraheemalsaghier/Desktop/Personal Projects/yolo/data/yolo_mot_dataset/labels/val.cache... 2851 images, 0 backgrounds, 0 corrupt: 100%|██████████| 2851/2851 [00:00<?, ?it/s]


Plotting labels to runs/detect/train2/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.000625, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to runs/detect/train2
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

      2/100         0G      1.899      1.212      1.296        123        640: 100%|██████████| 179/179 [19:17<00:00,  6.47s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 90/90 [05:42<00:00,  3.81s/it]


                   all       2851     245001      0.796      0.767      0.819      0.445

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      3/100         0G        1.7     0.9828      1.207        269        640: 100%|██████████| 179/179 [19:43<00:00,  6.61s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 90/90 [05:49<00:00,  3.88s/it]


                   all       2851     245001      0.827      0.799      0.854      0.494

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      4/100         0G      1.603     0.8913      1.163       2198        640:  92%|█████████▏| 165/179 [9:31:51<01:13,  5.22s/it]     

In [94]:
model = YOLO('./yolov8nbest.pt')

In [95]:
output_dir = os.path.join(OUTPUT, "yolov8nbest")
results = evaluate(model, MOT20DIRTEST, output_dir)

Resetting Tracker
Processing sequence: MOT20-04


MOT20-04: 100%|██████████| 2080/2080 [01:48<00:00, 19.11it/s]


Resetting Tracker
Processing sequence: MOT20-07


MOT20-07: 100%|██████████| 585/585 [00:26<00:00, 22.26it/s]


Resetting Tracker
Processing sequence: MOT20-08


MOT20-08: 100%|██████████| 806/806 [00:30<00:00, 26.79it/s]


Resetting Tracker
Processing sequence: MOT20-06


MOT20-06: 100%|██████████| 1008/1008 [00:38<00:00, 25.85it/s]


In [93]:
video_dir = os.path.join(OUTPUT,"videos")
create_video(model, img_files, SEQ_PATH, video_dir, "bestnano")

MOT20-08: 100%|██████████| 806/806 [01:05<00:00, 12.40it/s]
